<a href="https://colab.research.google.com/github/VladousSparrowous/HSE-homeworks/blob/main/BERTopic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BERTopic ДЗ

> Итак вот третье ДЗ. Вам дан датасет lenta3_lemat.csv. Это лематизированная версия известного датасета, в котором 10 тем. Так как там темы частично пересекаются то могут в принципе быть число тем от 7-10 штук. Ч то нужно сделать: Вам нужно сделать тематическое моделирование при помощи Bertopic на 30 тем, затем к этому тематическом моделированию применить процедуру ренормализации по аналогии с моделью plsa (все рассказывал и показывал на прошлом занятии). Ваша цель определить число тем на основе модели Bertopic.
Deadline сдачи дз 16 марта (до 12 часов ночи)






In [ ]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 14.0 MB/s eta 0:00:00


In [ ]:
from bertopic import BERTopic
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/lenta3_lemat.csv", header=None)
texts = df[0].tolist()

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [ ]:
print(*texts[:10], sep='\n')

жесткий санкция анонсировать совет безопасность осуществлять внимательно следить страна решительно замечать проведение тысяча километр тип исход новое тип воспринимать попытаться основатель вразрез совет безопасность оон минута сша район двигатель вероятно позволять нарушение спутник потребовать ситуация ким чен ын осуждать орбит несмотря военный документ значительный мера ракета республика испытание одобрять северокорейский разница сторона заявление нарушать наземный земля неудачный воздерживаться мирова сообщество регион достигать начало прошлое результат полигон северный корея национальный праздник ядерный оружие сен дальнейший масштабный аляска впоследствии покрытие слежение совбез баллистический ракета годовщина оон разрабатывать пхеньян попытка оценка кндр пуск межконтинентальный баллистический ракета случай проводить ряд предназначать вывести неделя производить целить лидер разрабатывать учение резолюция остер материал рождение технология подчеркиваться западный побережье дальне

In [ ]:
print(len(texts[0]))

1114


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
from sentence_transformers import SentenceTransformer


device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("DeepPavlov/rubert-base-cased-sentence", device=device)

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32
)

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/711M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Batches:   0%|          | 0/271 [00:00<?, ?it/s]

In [ ]:
topic_model = BERTopic(nr_topics=30, calculate_probabilities=True, language='russian')
topics, probs = topic_model.fit_transform(texts, embeddings)

In [ ]:
# @title ренормализация

phi_sparse = topic_model.c_tf_idf_
words = topic_model.vectorizer_model.get_feature_names_out()
topic_ids = topic_model.get_topic_info().Topic.tolist()

phi_df = pd.DataFrame(phi_sparse.T.toarray(), index=words)
phi_df.columns = topic_ids

if -1 in phi_df.columns:
    phi_df = phi_df.drop(columns=[-1])

phi_df = phi_df.div(phi_df.sum(axis=0), axis=1)

In [ ]:
# @title темы
import numpy as np

topic_probs = probs.sum(axis=0)
topic_probs /= topic_probs.sum()

threshold = 0.04 # порог
significant_topics = np.where(topic_probs >= threshold)[0]
significant_topics = significant_topics[np.argsort(topic_probs[significant_topics])[::-1]]

print(f"Реальные темы({len(significant_topics)}) и их топ слов:\n")
for t in significant_topics:
    weight = topic_probs[t]
    top_words = phi_df[t].sort_values(ascending=False).head(10)
    print(f"Topic {t}: weight={weight:.3f}")
    for word, prob in top_words.items():
        print(f"{word} ({prob:.3f})")
    print("-"*60)

Реальные темы(8) и их топ слов:

Topic 2: weight=0.172
россия (0.004)
страна (0.004)
президент (0.003)
украина (0.003)
отношение (0.003)
санкция (0.003)
российский (0.003)
украинский (0.002)
крым (0.002)
глава (0.002)
------------------------------------------------------------
Topic 1: weight=0.140
террористический (0.003)
происходить (0.003)
группировка (0.003)
суд (0.003)
уголовный (0.003)
сотрудник (0.003)
полиция (0.003)
задерживать (0.003)
город (0.002)
организация (0.002)
------------------------------------------------------------
Topic 6: weight=0.091
банк (0.007)
россия (0.006)
экономика (0.005)
пенсионный (0.005)
процент (0.005)
рубль (0.005)
экономический (0.004)
рынок (0.004)
миллиард (0.004)
российский (0.003)
------------------------------------------------------------
Topic 0: weight=0.064
женщина (0.006)
летний (0.004)
мужчина (0.003)
девушка (0.003)
ребенок (0.003)
полиция (0.002)
следственный (0.002)
девочка (0.002)
происходить (0.002)
дом (0.002)
-------------------